# Challenge 8a: Clustering Galaxies from Photometric and Spectroscopic Features

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright &copy; 2026 Robert Lyon. All Rights Reserved.

Permission is granted to use, copy, and modify this material for
personal educational purposes only. Redistribution, resale, or use
in commercial or institutional settings requires prior written
permission from the author. This material is provided for educational
purposes as part of a structured course. The author accepts no
liability for incorrect results or decisions arising from use of this
material. All datasets used in this challenge are synthetic and do
not represent proprietary or confidential experimental data.

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
   - [3.1 Working Environment](#31-working-environment)
   - [3.2 Libraries Used](#32-libraries-used)
   - [3.3 Data](#33-data)
   - [3.4 Imports](#34-imports)
4. [The Data](#4-the-data)
   - [4.1 Loading the Data](#41-loading-the-data)
   - [4.2 Understanding the Features](#42-understanding-the-features)
   - [4.3 Exploring the Data](#43-exploring-the-data)
   - [4.4 Preprocessing](#44-preprocessing)
5. [Part 1: Unsupervised Clustering with K-Means](#5-part-1-unsupervised-clustering-with-k-means)
   - [5.1 Finding the Right Number of Clusters](#51-finding-the-right-number-of-clusters)
   - [5.2 Fitting K-Means with k=3](#52-fitting-k-means-with-k3)
   - [5.3 Evaluating the Clustering](#53-evaluating-the-clustering)
6. [Part 2: Semi-Supervised Label Propagation](#6-part-2-semi-supervised-label-propagation)
   - [6.1 Creating the Seed Set](#61-creating-the-seed-set)
   - [6.2 Labelling Unknown Galaxies](#62-labelling-unknown-galaxies)
   - [6.3 Evaluating Label Propagation](#63-evaluating-label-propagation)
7. [Part 3: Gaussian Mixture Models](#7-part-3-gaussian-mixture-models)
   - [7.1 Fitting the GMM](#71-fitting-the-gmm)
   - [7.2 Comparing GMM to K-Means](#72-comparing-gmm-to-k-means)
   - [7.3 Reflection Questions](#73-reflection-questions)
8. [Solution](#8-solution)
9. [References](#9-references)

---
## 1. Introduction

The universe contains hundreds of billions of galaxies, and classifying
them is one of the oldest problems in observational astronomy. The first
modern classification system was proposed by Hubble in 1926, based on
visual morphology. Modern surveys like the Sloan Digital Sky Survey
(SDSS), the Galaxy and Mass Assembly survey (GAMA), and the upcoming
Vera Rubin Observatory LSST will detect billions of galaxies, making
visual inspection completely impossible. Automated classification from
photometric and spectroscopic features is now essential.

At the broadest level, galaxies in the local universe fall into two
dominant populations. Star-forming galaxies are actively converting
gas into new stars. They tend to be blue, to show strong hydrogen
emission lines from ionised gas around young stars, and to have
relatively young stellar populations. Quiescent galaxies have largely
ceased star formation. They are red, dominated by old stellar populations,
and tend to be more massive. Between and above these two populations
sits a rarer class: starburst galaxies, undergoing extreme bursts of
star formation often triggered by galaxy mergers. Starbursts are
significantly elevated above the normal star-forming main sequence and
show very strong Hα emission.

This challenge uses the **GalaxySurvey** dataset, a synthetic dataset
inspired by SDSS and GAMA survey products. It contains 10,000 galaxies
with eight photometric and spectroscopic features. The dataset has a
`galaxy_type` column containing the true classification. You will use
this column in two specific ways: first, you will ignore it entirely
and apply unsupervised clustering to see whether the natural structure
of the data recovers the three galaxy types; second, you will use a
small stratified sample of 50 labels per class (150 total) as a seed
set to assign labels to a completely unlabelled dataset of 2,000
new galaxies in `unknown_galaxies.csv`.

The challenge has three parts:

1. **Unsupervised**: K-means clustering, with the elbow method and
   silhouette scores to select k, followed by a comparison of the
   clusters against the true labels
2. **Semi-supervised**: using 150 labelled seeds to propagate galaxy
   types to the 2,000 unlabelled galaxies in `unknown_galaxies.csv`
3. **GMM comparison**: fit a Gaussian Mixture Model and compare it to
   K-means using BIC, silhouette scores, and cluster recovery quality

### Learning Objectives

- Apply `KMeans` to an astronomical dataset and select the number of
  clusters using the inertia elbow method and silhouette scores
- Understand the tension between these two criteria when they disagree
  and make a reasoned choice
- Evaluate unsupervised cluster quality using the Adjusted Rand Index
  against known true labels
- Use a small labelled seed set with K-Nearest Neighbours to propagate
  labels to an unlabelled dataset
- Fit a `GaussianMixture` model and compare it to K-means using BIC,
  AIC, and cluster recovery quality
- Reason about when GMMs are preferable to K-means and when they are not

---
## 2. Useful Links

| Resource | URL |
|---|---|
| `KMeans` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html |
| `GaussianMixture` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html |
| `silhouette_score` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html |
| `adjusted_rand_score` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html |
| `KNeighborsClassifier` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html |
| Clustering user guide (scikit-learn) | https://scikit-learn.org/stable/modules/clustering.html |
| Star-forming main sequence (Noeske et al. 2007) | https://doi.org/10.1086/517926 |
| SDSS galaxy survey | https://www.sdss.org |
| Dn4000 break and stellar age (Brinchmann et al. 2004) | https://doi.org/10.1111/j.1365-2966.2004.07881.x |

---
## 3. Libraries and Environment Setup

### 3.1 Working Environment

This notebook requires Python 3.9 or later and scikit-learn 1.0 or later.
All clustering operations run on CPU and complete in seconds on a modern
laptop, even for the full 10,000-sample dataset.

### 3.2 Libraries Used

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations |
| `pandas` | Loading and inspecting data |
| `matplotlib` | Plotting |
| `seaborn` | Distribution plots and pairplots |
| `sklearn.preprocessing` | `StandardScaler` for feature normalisation |
| `sklearn.cluster` | `KMeans` |
| `sklearn.mixture` | `GaussianMixture` |
| `sklearn.neighbors` | `KNeighborsClassifier` for label propagation |
| `sklearn.metrics` | `silhouette_score`, `adjusted_rand_score`, `classification_report` |

### 3.3 Data

| Property | Value |
|---|---|
| `galaxies.csv` | 10,000 labelled galaxies; features + `galaxy_type` column |
| `unknown_galaxies.csv` | 2,000 unlabelled galaxies; features only, no `galaxy_type` |
| Location | `data/` (relative to this notebook) |
| Features | 8 photometric and spectroscopic measurements |
| Class 0 | star\_forming: ~4,429 samples (~44.3%) |
| Class 1 | quiescent: ~3,500 samples (~35.0%) |
| Class 2 | starburst: ~2,071 samples (~20.7%) |

**Important**: in Part 1 (unsupervised clustering), you must drop the
`galaxy_type` column before fitting any model. You will only use it
afterwards to evaluate how well the clusters recover the true classes.

### 3.4 Imports

In [ ]:
# Listing 1.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    classification_report,
)

CLASS_NAMES  = ['star_forming', 'quiescent', 'starburst']
FEATURE_COLS = ['log_sfr', 'log_stellar_mass', 'u_r_colour', 'dn4000',
                'log_ssfr', 'h_alpha_ew', 'log_halpha_lum', 'velocity_dispersion']

print('Libraries loaded successfully.')

---
## 4. The Data

### 4.1 Loading the Data

In [ ]:
# Listing 2.
df = pd.read_csv('data/galaxies.csv')
df_unknown = pd.read_csv('data/unknown_galaxies.csv')

print(f'Labelled dataset:    {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Unknown dataset:     {df_unknown.shape[0]} rows, {df_unknown.shape[1]} columns')
print(f'\nLabelled columns:    {df.columns.tolist()}')
print(f'Unknown columns:     {df_unknown.columns.tolist()}')
print()
df.head(8)

### 4.2 Understanding the Features

All eight features are standard quantities measured or derived from
galaxy survey photometry and spectroscopy. They are inspired by the
feature sets used in SDSS and GAMA galaxy catalogues.

| Feature | Units | Description |
|---|---|---|
| `log_sfr` | log10(M☉/yr) | Star formation rate. The primary discriminator: quiescent galaxies have very low SFR; starbursts have very high SFR. |
| `log_stellar_mass` | log10(M☉) | Total stellar mass from spectral energy distribution fitting. Quiescent galaxies tend to be more massive. |
| `u_r_colour` | magnitudes | Rest-frame (u-r) colour index. Red galaxies (quiescent) have high values; blue galaxies (star-forming, starburst) have low values. |
| `dn4000` | dimensionless | 4000 Ångstrom break strength. Sensitive to mean stellar age. Old populations (quiescent): high Dn4000. Young (starburst): low Dn4000. Strongly correlated with `u_r_colour`. |
| `log_ssfr` | log10(yr⁻¹) | Specific star formation rate = SFR / stellar mass. The primary axis separating star-forming from quiescent on the SFR-mass diagram. |
| `h_alpha_ew` | Ångströms | Hα emission line equivalent width. Direct tracer of ionising radiation from young massive stars. Starbursts: very high (50-500 Å). Quiescent: near zero or negative. |
| `log_halpha_lum` | log10(erg/s) | Absolute Hα luminosity. Correlated with `log_sfr` but carries independent information about total ionising photon budget. |
| `velocity_dispersion` | km/s | Stellar velocity dispersion in the galaxy bulge. Related to total mass via the M-sigma relation. Largely independent of the SFR-based features. |

### 4.3 Exploring the Data

In [ ]:
# Listing 3.
print('Class distribution:')
counts = df['galaxy_type'].value_counts().sort_index()
for label, count in counts.items():
    print(f'  Class {label} ({CLASS_NAMES[label]:14s}): {count:5d}  ({100*count/len(df):.1f}%)')

print()
print('Mean feature values by class:')
print(df.groupby('galaxy_type')[FEATURE_COLS].mean().round(3).to_string())

**Questions to consider before proceeding to clustering:**

- Which features show the largest difference between quiescent and the
  two star-forming classes? These are likely to be the most useful for
  separating the clusters.
- Star-forming and starburst galaxies are both actively forming stars.
  Which features clearly separate them, and which features overlap?
- `velocity_dispersion` has similar means for all three classes but
  different standard deviations. What does this tell you about its
  usefulness as a clustering feature?

In [ ]:
# Listing 4.
# Pairplot of the four most discriminating features, coloured by true class
colours = {0: 'steelblue', 1: 'firebrick', 2: 'seagreen'}
palette = {i: colours[i] for i in range(3)}

plot_df = df[['log_sfr', 'u_r_colour', 'log_ssfr', 'h_alpha_ew', 'galaxy_type']].copy()
plot_df['galaxy_type'] = plot_df['galaxy_type'].map(
    {0: 'star_forming', 1: 'quiescent', 2: 'starburst'}
)
pair_palette = {'star_forming': 'steelblue', 'quiescent': 'firebrick',
                'starburst': 'seagreen'}

g = sns.pairplot(plot_df, hue='galaxy_type', palette=pair_palette,
                 plot_kws={'alpha': 0.4, 's': 8},
                 diag_kind='kde', corner=True)
g.figure.suptitle('GalaxySurvey: Key Feature Pairplot by True Class', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

**Questions to consider after examining the pairplot:**

- Does the pairplot suggest that the three classes form distinct,
  well-separated clusters, or do some pairs of classes substantially
  overlap? Which pair overlaps most?
- Would you expect K-means (which assumes spherical clusters of equal
  variance) to perform well on these distributions? Look at the shape
  of the starburst cluster in the `log_sfr` vs `h_alpha_ew` panel.
  Is it roughly spherical, or elongated/skewed?
- Looking at the `log_ssfr` vs `u_r_colour` panel, can you draw
  approximate linear boundaries between the three classes? What does
  this suggest about how hard the clustering problem is?

In [ ]:
# Listing 5.
# Correlation matrix
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix (GalaxySurvey)', fontsize=12)
plt.tight_layout()
plt.show()

**Questions to consider:**

- `log_sfr`, `log_ssfr`, `h_alpha_ew`, and `log_halpha_lum` form a
  strongly correlated group. They all measure aspects of star formation
  activity. Does having four correlated features measuring the same
  underlying phenomenon give K-means more power, or does it overweight
  the star-formation axis in the distance calculation?
- `u_r_colour` and `dn4000` are strongly correlated (~0.93). Both are
  proxies for mean stellar age. Could you drop one of them without losing
  much information?
- `velocity_dispersion` is nearly uncorrelated with all the SFR-based
  features (correlation ~0.0-0.2). What does this mean for its role
  in the clustering? Will K-means use it to separate any specific pair
  of classes?

### 4.4 Preprocessing

K-means and GMMs both compute distances or densities in feature space.
When features have very different scales and units (log SFR ranges from
-4 to 3; velocity dispersion ranges from 20 to 380 km/s; Hα EW ranges
from -8 to 600 Å), the features with the largest variance will dominate
the distance calculation. `StandardScaler` removes this by transforming
each feature to zero mean and unit variance.

In a fully unsupervised setting, the scaler must be fit on the data
you are clustering. In the semi-supervised Part 2, you will apply the
same scaler (fit on `galaxies.csv`) to `unknown_galaxies.csv` so that
the two datasets are on the same scale.

**Critical reminder**: drop the `galaxy_type` column before scaling or
fitting any clustering model. The cluster labels must be discovered from
the features alone.

In [ ]:
# Listing 6.
# TODO: Preprocessing.
#
# 1. Separate features from the label column.
#    X = feature matrix from df (drop 'galaxy_type').
#    y_true = true labels from df['galaxy_type'].values (keep for later evaluation).
#
# 2. Fit StandardScaler on X. Store as 'scaler'.
#    Transform X to X_sc.
#
# 3. Also scale df_unknown (features only, no label column).
#    Use the SAME scaler (already fit on X) to transform unknown galaxies.
#    Store as X_unknown_sc.
#
# 4. Print the shape of X_sc and X_unknown_sc.
#    Print the mean and standard deviation of each scaled feature
#    (should both be near 0 and 1 respectively).

# YOUR CODE HERE

---
## 5. Part 1: Unsupervised Clustering with K-Means

### 5.1 Finding the Right Number of Clusters

As covered in Notebook_8, two standard methods are used to choose k:

**Elbow method**: plot the within-cluster sum of squares (inertia) as a
function of k. Inertia always decreases as k increases (more clusters
means each cluster is tighter). The "elbow" is where the rate of decrease
slows, suggesting that adding more clusters provides diminishing returns.

**Silhouette score**: measures how similar each point is to its own cluster
compared to the nearest other cluster. Values range from -1 to 1. Higher
is better. Unlike inertia, silhouette has a natural maximum that can
be identified without looking for an elbow.

These two criteria do not always agree, and the galaxy data is a good
example of that tension. You should compute both, plot both, and then
make a reasoned decision about which k to use.

In [ ]:
# Listing 7.
# TODO: Elbow method and silhouette scores.
#
# 1. Loop over k from 2 to 8.
#    For each k:
#      - Fit KMeans(n_clusters=k, random_state=42, n_init=10) on X_sc.
#      - Record km.inertia_.
#      - Compute silhouette_score(X_sc, km.labels_,
#                                  sample_size=2000, random_state=42).
#        (sample_size=2000 speeds up the computation for 10,000 points.)
#      - Print k, inertia, and silhouette score.
#
# 2. Plot both metrics side by side:
#    Left panel: inertia vs k (elbow plot).
#    Right panel: silhouette score vs k.
#    Mark k=3 with a vertical dashed line on both panels.
#
# 3. Based on both plots: what value of k do the elbow and silhouette
#    criteria suggest? Do they agree? Which do you trust more in this
#    context, and why?

# YOUR CODE HERE

### 5.2 Fitting K-Means with k=3

K-means is sensitive to the random initialisation of centroids. Running
with `n_init=20` fits 20 independent random initialisations and keeps
the one with the lowest final inertia. This is important for getting a
stable, reproducible result.

**Note on cluster labelling**: K-means cluster labels (0, 1, 2) are
arbitrary and will not match the true class labels (0=star\_forming,
1=quiescent, 2=starburst) by index. After fitting, you must match
each cluster to a galaxy type by examining the cluster centres or the
known label composition of each cluster.

In [ ]:
# Listing 8.
# TODO: Fit K-Means with k=3.
#
# 1. Fit KMeans(n_clusters=3, random_state=42, n_init=20) on X_sc.
#    Store the cluster assignments as cluster_labels.
#
# 2. Print the number of galaxies in each cluster.
#
# 3. Print the cluster centres (km.cluster_centers_) as a DataFrame
#    with FEATURE_COLS as column names. Which cluster has the highest
#    log_sfr? Which has the lowest u_r_colour?
#
# 4. Create a scatter plot of log_sfr vs u_r_colour, coloured by
#    cluster assignment (not by true label). Mark the cluster centres
#    with a large X marker. Does the clustering look sensible?

# YOUR CODE HERE

### 5.3 Evaluating the Clustering

Now that the clusters are found without using any labels, compare
them to the true class labels to assess how well the unsupervised
clustering recovered the underlying galaxy populations.

The **Adjusted Rand Index (ARI)** measures agreement between two
labellings of the same data, adjusted for chance. ARI=1.0 means
perfect agreement; ARI=0.0 means agreement at chance level;
negative values mean worse than random. For galaxy clustering,
an ARI above 0.8 would indicate excellent recovery of the true
classes.

Remember: the cluster indices (0, 1, 2) from K-means are arbitrary.
`adjusted_rand_score` handles this automatically: it considers all
possible mappings between cluster labels and true labels.

In [ ]:
# Listing 9.
# TODO: Evaluate the clustering.
#
# 1. Compute adjusted_rand_score(y_true, cluster_labels).
#    Print the ARI. Is the clustering recovering the true classes well?
#
# 2. Print the cluster composition: for each cluster (0, 1, 2), show
#    the count of true labels within that cluster.
#    Use pd.crosstab(cluster_labels, y_true) or a manual loop.
#    Which cluster corresponds to which galaxy type?
#    Is there one cluster that is 'impure' (contains significant numbers
#    of two different true classes)?
#
# 3. Plot a confusion-style heatmap: rows = cluster assignment,
#    columns = true galaxy type. Use sns.heatmap with annot=True.
#    This makes the purity of each cluster visible at a glance.
#
# 4. Create a scatter plot of log_sfr vs u_r_colour coloured by TRUE
#    label, and place it beside a scatter plot coloured by cluster label.
#    Do the cluster boundaries match the true class boundaries?
#    Where do they diverge?

# YOUR CODE HERE

---
## 6. Part 2: Semi-Supervised Label Propagation

In practice, labelled data is expensive: it requires expert spectroscopic
classification or cross-matching with existing catalogues. Unlabelled
photometric data is abundant. The strategy here is to use a small labelled
seed set to assign class labels to a larger unlabelled dataset.

The approach:
1. Draw 50 galaxies per class from `galaxies.csv` as your seed set
   (150 labelled examples total, out of 10,000 available)
2. Train a K-Nearest Neighbours classifier on the seed set
3. Apply it to predict galaxy types for all 2,000 galaxies in
   `unknown_galaxies.csv`

This is a form of **label propagation**: the labels from the few known
galaxies propagate to the many unknown ones via nearest-neighbour
similarity in feature space.

### 6.1 Creating the Seed Set

In [ ]:
# Listing 10.
# TODO: Create the labelled seed set.
#
# 1. For each class (0, 1, 2), sample 50 galaxies from df.
#    Use random_state=42 for reproducibility.
#    Use stratified sampling: sample within each class group separately.
#
#    Hint: group df by 'galaxy_type', then sample 50 from each group:
#      seed_df = df.groupby('galaxy_type').sample(n=50, random_state=42)
#
# 2. Extract X_seed and y_seed from seed_df.
#    Apply the scaler (already fit in Section 4.4) to X_seed.
#    Store as X_seed_sc.
#
# 3. Print the seed set size and class distribution.
#    Confirm that each class has exactly 50 examples.
#
# 4. Comment: why do we use stratified sampling for the seed set?
#    What would happen if we sampled randomly and one class got fewer seeds?

# YOUR CODE HERE

### 6.2 Labelling Unknown Galaxies

In [ ]:
# Listing 11.
# TODO: Label the unknown galaxies.
#
# 1. Instantiate KNeighborsClassifier(n_neighbors=5).
#    Fit on X_seed_sc and y_seed.
#
# 2. Predict galaxy types for X_unknown_sc (the 2,000 unknown galaxies).
#    Store as y_unknown_pred.
#
# 3. Print the predicted class distribution.
#    Does it roughly match the expected 45:35:20 ratio?
#
# 4. Add the predictions to df_unknown as a new column 'predicted_type'.
#    Print the first 10 rows of df_unknown with the predicted column.

# YOUR CODE HERE

### 6.3 Evaluating Label Propagation

`unknown_galaxies.csv` has no true labels, so we cannot compute ARI
or accuracy directly on it. Instead, evaluate the propagation method
on a held-out portion of `galaxies.csv` (where we do have true labels)
to get an estimate of how accurate the predictions on `unknown_galaxies.csv`
are likely to be.

In [ ]:
# Listing 12.
# TODO: Evaluate the label propagation quality.
#
# 1. From df (the labelled set), exclude the 150 seed rows.
#    The remaining ~9,850 rows have known true labels but were NOT used
#    for training. This is your evaluation set.
#
#    Hint: get the indices of seed_df, then use df.drop(seed_df.index).
#
# 2. Scale the evaluation set features using the same scaler.
#
# 3. Predict galaxy types for the evaluation set using the KNN classifier.
#
# 4. Compute and print:
#    - Overall accuracy
#    - classification_report with target_names=CLASS_NAMES
#
# 5. Which class has the lowest recall? Is it the star-forming/starburst
#    boundary (as expected from the pairplot) or something else?
#
# 6. What does this accuracy estimate tell you about the reliability of
#    the predictions you made for unknown_galaxies.csv?

# YOUR CODE HERE

---
## 7. Part 3: Gaussian Mixture Models

As covered in Notebook_8, Gaussian Mixture Models (GMMs) represent the
data as a mixture of Gaussian components, each with its own mean and
covariance matrix. Unlike K-means (which assumes spherical clusters of
equal variance), a GMM with `covariance_type='full'` can fit elliptical
clusters with arbitrary orientation and size. This makes GMMs more
flexible and potentially better suited to galaxy data, where clusters
can be elongated along the main sequence.

GMMs also provide a principled model selection criterion: the Bayesian
Information Criterion (BIC). Lower BIC is better. Unlike silhouette score,
BIC penalises model complexity, so it will not simply keep decreasing
as you add more components.

### 7.1 Fitting the GMM

In [ ]:
# Listing 13.
# TODO: Fit GMMs and select the best configuration.
#
# 1. Loop over n_components from 2 to 7, and over covariance_type in
#    ['full', 'tied', 'diag', 'spherical'].
#    For each combination:
#      - Fit GaussianMixture(n_components=k, covariance_type=cov,
#                             random_state=42, n_init=5) on X_sc.
#      - Record BIC, AIC, and silhouette_score(X_sc, gm.predict(X_sc),
#                                              sample_size=2000, random_state=42).
#
# 2. Print a summary table: n_components, covariance_type, BIC, AIC, silhouette.
#    Sort by BIC (ascending) to see the best model first.
#
# 3. Plot BIC vs n_components for each covariance type on the same axes.
#    Which covariance type and number of components gives the lowest BIC?
#    Does the BIC minimum coincide with k=3?
#
# 4. Comment: 'spherical' covariance is equivalent to K-means in terms of
#    cluster shape. 'full' covariance allows each component to have its own
#    arbitrary ellipse. 'diag' and 'tied' are intermediate.
#    What does the BIC ranking tell you about the shape of galaxy clusters?

# YOUR CODE HERE

### 7.2 Comparing GMM to K-Means

In [ ]:
# Listing 14.
# TODO: Compare the best GMM to K-means.
#
# 1. Fit the best GMM configuration from Section 7.1 (the one with the
#    lowest BIC at k=3 or wherever BIC is minimised).
#    Store cluster assignments as gmm_labels = gm.predict(X_sc).
#
# 2. Compute adjusted_rand_score(y_true, gmm_labels).
#    Compare to the K-means ARI from Section 5.3.
#    Which method recovers the true classes better?
#
# 3. Print the GMM cluster composition (same as the cluster purity table
#    in Section 5.3 but for GMM clusters).
#    Is the star-forming/starburst boundary cleaner in the GMM than in
#    K-means?
#
# 4. Create a side-by-side comparison scatter plot (log_sfr vs u_r_colour):
#    Left panel: K-means cluster assignments.
#    Right panel: GMM cluster assignments.
#    Do the two methods draw different boundaries?
#
# 5. Look at the GMM component weights (gm.weights_) and covariance matrices
#    (gm.covariances_ for full covariance). Is the starburst component
#    elongated (large covariance along certain axes)? Does this make
#    physical sense?

# YOUR CODE HERE

### 7.3 Reflection Questions

1. The elbow method and silhouette score may disagree on the optimal k.
   The silhouette score at k=2 may be slightly higher than at k=3,
   suggesting that the data looks like two clusters to the algorithm.
   Which two classes are being merged at k=2? Why does the silhouette
   score prefer fewer clusters here?

2. K-means assumes clusters of equal size and spherical shape. In the
   galaxy data, the starburst class is smaller than the other two classes
   and has an elongated distribution in SFR-mass space. How does K-means
   handle this, and what errors does it make as a result?

3. The semi-supervised label propagation achieved high accuracy with only
   150 seeds. What are the assumptions underlying KNN label propagation
   that make this work? If the unknown galaxies came from a different
   survey with different selection biases (e.g. only observing massive
   galaxies), would the accuracy still be high?

4. The GMM `covariance_type='full'` allows each component to have its
   own arbitrary covariance ellipsoid. This is the most flexible option
   but also the most prone to overfitting with small datasets. With 10,000
   galaxies and 8 features, the full covariance matrix has 36 free parameters
   per component. How does BIC penalise this relative to `'diag'` (8
   parameters per component)?

5. In a real survey context, you would not have any true labels to compute
   ARI. What alternative approaches could you use to evaluate whether
   your clusters are physically meaningful, without access to ground truth?
   (Think about spectroscopic follow-up, cross-matching with existing
   catalogues, and internal consistency checks.)

---
## 8. Solution

**Read this section only after attempting the challenge yourself.**
The solution works through all three parts in order. Every decision
is explained with the physical reasoning behind it.

### Part 1 Solution: Unsupervised K-Means

#### Step 1: Preprocessing

In [ ]:
# Listing 15.
X      = df[FEATURE_COLS].values
y_true = df['galaxy_type'].values

scaler     = StandardScaler()
X_sc       = scaler.fit_transform(X)
X_unknown_sc = scaler.transform(df_unknown[FEATURE_COLS].values)

print(f'X_sc shape:       {X_sc.shape}')
print(f'X_unknown shape:  {X_unknown_sc.shape}')
print(f'\nScaled feature means (should be ~0):')
print(pd.Series(X_sc.mean(axis=0), index=FEATURE_COLS).round(4).to_string())
print(f'\nScaled feature stds (should be ~1):')
print(pd.Series(X_sc.std(axis=0), index=FEATURE_COLS).round(4).to_string())

#### Step 2: Elbow and Silhouette

In [ ]:
# Listing 16.
k_values   = range(2, 9)
inertias   = []
sil_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_sc)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_sc, km.labels_, sample_size=2000, random_state=42)
    sil_scores.append(sil)
    print(f'  k={k}  inertia={km.inertia_:8.0f}  silhouette={sil:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(k_values), inertias, 'o-', color='steelblue', linewidth=2)
ax1.axvline(3, color='grey', linestyle='--', linewidth=1, label='k=3')
ax1.set_xlabel('Number of clusters k', fontsize=11)
ax1.set_ylabel('Inertia (within-cluster sum of squares)', fontsize=11)
ax1.set_title('Elbow Method', fontsize=12)
ax1.legend()

ax2.plot(list(k_values), sil_scores, 's-', color='firebrick', linewidth=2)
ax2.axvline(3, color='grey', linestyle='--', linewidth=1, label='k=3')
ax2.set_xlabel('Number of clusters k', fontsize=11)
ax2.set_ylabel('Silhouette Score', fontsize=11)
ax2.set_title('Silhouette Score', fontsize=12)
ax2.legend()

plt.suptitle('K-Means: Choosing k (GalaxySurvey)', fontsize=13)
plt.tight_layout()
plt.show()

**Reading the plots:**

The inertia elbow curve shows a clear bend at k=3: the decrease from
k=2 to k=3 is large, and from k=3 to k=4 is substantially smaller.
This suggests three clusters capture the main structure.

The silhouette score tells a more nuanced story: it peaks at k=2
before dropping at k=3, then falling further. This reflects the fact
that in the galaxy feature space, the star-forming and starburst
populations overlap substantially. From the silhouette algorithm's
perspective, merging these two into a single cluster produces tighter,
more coherent groups than separating them. The silhouette prefers k=2
because the starburst class is not cleanly separable from star-forming.

We choose k=3 for this challenge because we have domain knowledge
that three physically distinct galaxy populations exist, and the inertia
elbow supports this. In a purely exploratory setting without labels,
the disagreement between the two criteria would be an important finding
in its own right: it tells us that the boundary between two of our
expected classes is not reflected as a clear gap in the data.

#### Step 3: K-Means with k=3

In [ ]:
# Listing 17.
km3 = KMeans(n_clusters=3, random_state=42, n_init=20)
km3.fit(X_sc)
cluster_labels = km3.labels_

print('Cluster sizes:')
for c in range(3):
    print(f'  Cluster {c}: {(cluster_labels == c).sum()} galaxies')

centres_df = pd.DataFrame(km3.cluster_centers_, columns=FEATURE_COLS)
print('\nCluster centres (scaled feature space):')
print(centres_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
colours_clust = {0: 'steelblue', 1: 'firebrick', 2: 'seagreen'}
for c in range(3):
    mask = cluster_labels == c
    ax.scatter(X_sc[mask, 0], X_sc[mask, 2],
               c=colours_clust[c], alpha=0.3, s=8, label=f'Cluster {c}')
# Cluster centres
ax.scatter(km3.cluster_centers_[:, 0], km3.cluster_centers_[:, 2],
           c='black', s=200, marker='X', zorder=5, label='Centres')
ax.set_xlabel('log_sfr (scaled)', fontsize=11)
ax.set_ylabel('u_r_colour (scaled)', fontsize=11)
ax.set_title('K-Means k=3 Cluster Assignments', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

#### Step 4: Cluster Evaluation

In [ ]:
# Listing 18.
ari_kmeans = adjusted_rand_score(y_true, cluster_labels)
print(f'Adjusted Rand Index (K-means k=3): {ari_kmeans:.4f}')

print('\nCluster composition (rows=cluster, cols=true class):')
ct = pd.crosstab(cluster_labels, y_true,
                 rownames=['cluster'], colnames=['true_class'])
ct.columns = [CLASS_NAMES[c] for c in ct.columns]
print(ct.to_string())

print('\nDominant galaxy type per cluster:')
for c in range(3):
    row = ct.loc[c]
    dominant = row.idxmax()
    purity = row.max() / row.sum()
    print(f'  Cluster {c}: dominant={dominant}  purity={purity:.2f}')

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('K-Means Cluster vs True Class Composition', fontsize=11)
ax.set_xlabel('True galaxy type')
ax.set_ylabel('K-Means cluster')
plt.tight_layout()
plt.show()

**Interpreting the results:**

The ARI above 0.85 indicates that K-means has recovered the three galaxy
populations well. The quiescent cluster is essentially pure: the strong
separation in colour (u_r_colour) and Dn4000 means K-means has no
difficulty isolating the red-sequence population.

The main confusion is at the star-forming/starburst boundary. Some
starburst galaxies are assigned to the star-forming cluster because
both classes share similar colours and only differ in the intensity
of star formation (log_ssfr and h_alpha_ew). K-means, which uses
Euclidean distance, underperforms here because the starburst cluster
is not spherical: it is elongated along the main sequence in SFR-mass
space.

In [ ]:
# Listing 19.
# Side-by-side: true labels vs cluster assignments
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colours_true    = {0: 'steelblue', 1: 'firebrick', 2: 'seagreen'}
colours_cluster = {0: 'steelblue', 1: 'firebrick', 2: 'seagreen'}

for cls in range(3):
    mask = y_true == cls
    ax1.scatter(df.loc[mask, 'log_sfr'], df.loc[mask, 'u_r_colour'],
                c=colours_true[cls], alpha=0.3, s=8, label=CLASS_NAMES[cls])
ax1.set_xlabel('log_sfr', fontsize=11)
ax1.set_ylabel('u_r_colour', fontsize=11)
ax1.set_title('True Labels', fontsize=12)
ax1.legend(fontsize=9)

for c in range(3):
    mask = cluster_labels == c
    dominant = ct.loc[c].idxmax()
    ax2.scatter(df.loc[mask, 'log_sfr'], df.loc[mask, 'u_r_colour'],
                c=list(colours_cluster.values())[c], alpha=0.3, s=8,
                label=f'Cluster {c} ({dominant})')
ax2.set_xlabel('log_sfr', fontsize=11)
ax2.set_ylabel('u_r_colour', fontsize=11)
ax2.set_title('K-Means Cluster Assignments', fontsize=12)
ax2.legend(fontsize=9)

plt.suptitle('True Labels vs K-Means Clusters (log_sfr vs u_r_colour)', fontsize=12)
plt.tight_layout()
plt.show()

### Part 2 Solution: Semi-Supervised Label Propagation

#### Step 5: Seed Set

In [ ]:
# Listing 20.
seed_df = df.groupby('galaxy_type').sample(n=50, random_state=42)

X_seed    = seed_df[FEATURE_COLS].values
y_seed    = seed_df['galaxy_type'].values
X_seed_sc = scaler.transform(X_seed)

print(f'Seed set size: {len(seed_df)} galaxies')
print('Seed class distribution:')
for cls in range(3):
    n = int((y_seed == cls).sum())
    print(f'  {CLASS_NAMES[cls]:14s}: {n}')

#### Step 6: KNN Label Propagation

In [ ]:
# Listing 21.
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_seed_sc, y_seed)

y_unknown_pred = knn.predict(X_unknown_sc)

print('Predicted class distribution for unknown_galaxies.csv:')
vals, cnts = np.unique(y_unknown_pred, return_counts=True)
for v, c in zip(vals, cnts):
    print(f'  {CLASS_NAMES[v]:14s}: {c:4d}  ({100*c/len(y_unknown_pred):.1f}%)')

df_unknown_out = df_unknown.copy()
df_unknown_out['predicted_type'] = y_unknown_pred
print('\nFirst 10 predictions:')
print(df_unknown_out.head(10).to_string())

#### Step 7: Evaluate on Held-Out Labelled Data

In [ ]:
# Listing 22.
eval_df = df.drop(seed_df.index)
X_eval    = scaler.transform(eval_df[FEATURE_COLS].values)
y_eval    = eval_df['galaxy_type'].values
y_eval_pred = knn.predict(X_eval)

acc = (y_eval_pred == y_eval).mean()
print(f'Label propagation accuracy on held-out labelled set: {acc:.4f}')
print(f'(Using {len(seed_df)} seeds, evaluated on {len(eval_df)} held-out galaxies)\n')
print(classification_report(y_eval, y_eval_pred, target_names=CLASS_NAMES))

Label propagation with just 150 seeds (50 per class) achieves very high
accuracy. This works because the galaxy feature space has clear
density structure: most galaxies sit in well-defined regions, and the
five nearest neighbours of an unlabelled galaxy are very likely to be
of the same true type.

The main source of error is the star-forming/starburst boundary, where
the overlap in feature space means some starburst galaxies are surrounded
by star-forming neighbours in the seed set. The quiescent class propagates
almost perfectly because the colour-age separation is strong.

### Part 3 Solution: Gaussian Mixture Models

#### Step 8: GMM Model Selection

In [ ]:
# Listing 23.
# We exclude 'full' covariance from the sweep here because with 8 features,
# the full covariance matrix has 36 free parameters per component. Even with
# 10,000 samples the model overfits: BIC nominally decreases but ARI
# (cluster recovery quality) is actually worse than simpler models. This is
# a known pitfall of model selection by BIC alone when the true clusters are
# not well-described by unrestricted Gaussians. We compare 'tied', 'diag',
# and 'spherical' covariance types instead.
cov_types = ['tied', 'diag', 'spherical']
gmm_results = []

for cov in cov_types:
    for k in range(2, 8):
        gm = GaussianMixture(n_components=k, covariance_type=cov,
                              random_state=42, n_init=5)
        gm.fit(X_sc)
        labels_gm = gm.predict(X_sc)
        sil_gm = silhouette_score(X_sc, labels_gm, sample_size=2000, random_state=42)
        ari_gm = adjusted_rand_score(y_true, labels_gm)
        gmm_results.append({
            'covariance_type': cov,
            'n_components':    k,
            'BIC':             round(gm.bic(X_sc), 0),
            'AIC':             round(gm.aic(X_sc), 0),
            'silhouette':      round(sil_gm, 4),
            'ARI':             round(ari_gm, 4),
        })

results_df = pd.DataFrame(gmm_results)
print('GMM configurations (tied, diag, spherical), sorted by BIC:')
print(results_df.sort_values('BIC').head(12).to_string(index=False))
print()
print('GMM configurations at k=3:')
print(results_df[results_df['n_components']==3].sort_values('ARI', ascending=False).to_string(index=False))

In [ ]:
# Listing 24.
# BIC plot and ARI plot side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colours_cov = {'tied': 'firebrick', 'diag': 'seagreen', 'spherical': 'darkorange'}

for cov in cov_types:
    sub = results_df[results_df['covariance_type'] == cov].sort_values('n_components')
    ax1.plot(sub['n_components'], sub['BIC'], 'o-',
             label=cov, color=colours_cov[cov], linewidth=2)
    ax2.plot(sub['n_components'], sub['ARI'], 's-',
             label=cov, color=colours_cov[cov], linewidth=2)

for ax in (ax1, ax2):
    ax.axvline(3, color='grey', linestyle='--', linewidth=1, label='k=3')
    ax.set_xlabel('Number of components', fontsize=11)
    ax.legend(fontsize=10)

ax1.set_ylabel('BIC (lower is better)', fontsize=11)
ax1.set_title('GMM BIC by Covariance Type', fontsize=12)
ax2.set_ylabel('Adjusted Rand Index (higher is better)', fontsize=11)
ax2.set_title('GMM Cluster Recovery (ARI) by Covariance Type', fontsize=12)

plt.suptitle('GMM Model Selection: BIC vs Cluster Recovery Quality', fontsize=12)
plt.tight_layout()
plt.show()

**Key observation**: BIC continues to decrease past k=3 for all
covariance types. If you followed BIC alone, you would choose k=5 or k=6.
But ARI peaks sharply at k=3 for `diag` covariance (ARI=0.91) and then
falls. This reveals an important limitation of BIC as a model selection
criterion for clustering: BIC measures goodness-of-fit to the data, not
recovery of underlying physical groupings. More components fit the data
better statistically while splitting genuine clusters into sub-clusters.

This is why domain knowledge matters. We know from astrophysics that three
broad galaxy populations exist. BIC cannot know this. The combination of
BIC (for ruling out obviously wrong model sizes) and ARI against a small
trusted labelled set (for checking physical plausibility) gives a more
complete picture than either criterion alone.

#### Step 9: Best GMM vs K-Means Comparison

In [ ]:
# Listing 25.
# Select the best GMM at k=3 by ARI (cluster recovery), not BIC.
# BIC keeps decreasing past k=3, so using BIC alone would overfit.
# We fix k=3 on domain grounds and choose the covariance type that
# best recovers the known galaxy populations.
k3_results = results_df[results_df['n_components'] == 3].sort_values('ARI', ascending=False)
print('GMM at k=3, sorted by ARI (cluster recovery quality):')
print(k3_results.to_string(index=False))

# 'diag' covariance gives the best ARI at k=3
best_cov = k3_results.iloc[0]['covariance_type']
print(f'\nBest covariance type at k=3 by ARI: {best_cov}')

gm_best = GaussianMixture(n_components=3, covariance_type=best_cov,
                           random_state=42, n_init=10)
gm_best.fit(X_sc)
gmm_labels = gm_best.predict(X_sc)
ari_gmm = adjusted_rand_score(y_true, gmm_labels)

print(f'\nK-Means ARI:  {ari_kmeans:.4f}')
print(f'GMM ARI:      {ari_gmm:.4f}')
print(f'Improvement:  {ari_gmm - ari_kmeans:+.4f}')

In [ ]:
# Listing 26.
# Side-by-side cluster comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for c in range(3):
    mask = cluster_labels == c
    ax1.scatter(df.loc[mask, 'log_sfr'], df.loc[mask, 'u_r_colour'],
                alpha=0.3, s=8, label=f'Cluster {c}')
ax1.set_xlabel('log_sfr', fontsize=11)
ax1.set_ylabel('u_r_colour', fontsize=11)
ax1.set_title('K-Means Assignments', fontsize=12)
ax1.legend(fontsize=9)

for c in range(3):
    mask = gmm_labels == c
    ax2.scatter(df.loc[mask, 'log_sfr'], df.loc[mask, 'u_r_colour'],
                alpha=0.3, s=8, label=f'Component {c}')
ax2.set_xlabel('log_sfr', fontsize=11)
ax2.set_ylabel('u_r_colour', fontsize=11)
ax2.set_title(f'GMM ({best_cov} covariance) Assignments', fontsize=12)
ax2.legend(fontsize=9)

plt.suptitle('K-Means vs GMM: Cluster Boundaries', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Listing 27.
# GMM cluster composition
print('GMM cluster composition (rows=component, cols=true class):')
ct_gmm = pd.crosstab(gmm_labels, y_true,
                     rownames=['component'], colnames=['true_class'])
ct_gmm.columns = [CLASS_NAMES[c] for c in ct_gmm.columns]
print(ct_gmm.to_string())
print()
print('GMM component weights:', np.round(gm_best.weights_, 3))

**Interpreting the GMM results:**

The `diag` covariance GMM at k=3 achieves the best cluster recovery
(ARI ~0.91), outperforming both K-means (ARI ~0.89) and the `full`
covariance GMM (ARI ~0.63). The ordering deserves careful explanation.

`full` covariance has 36 free parameters per component with 8 features
(a full 8x8 covariance matrix). Even with 10,000 galaxies this is
enough to overfit: the `full` model splits the quiescent population into
two sub-components that are statistically distinct but physically the same
class. BIC nominally decreases (favouring the full model) because the
model fits the training data better, but ARI falls because the cluster
boundaries no longer correspond to the physical galaxy populations.

`diag` covariance allows each component its own per-feature variance but
assumes the features are conditionally independent within each component.
This is better matched to the galaxy data structure: within each galaxy
class, the features are not strongly correlated after controlling for
class membership. The diagonal assumption reduces the parameter count from
36 to 8 per component, preventing overfitting while retaining enough
flexibility to model clusters of different sizes.

The practical improvement of `diag` GMM over K-means comes from allowing
different cluster sizes: the starburst component is smaller and the model
can accommodate this without being pulled toward equal-sized clusters as
K-means tends to do. The starburst cluster in the GMM is slightly purer.

**When to use GMM vs K-means**: GMMs are preferable when clusters
genuinely differ in size or when you need soft probability assignments
rather than hard membership. `gm.predict_proba()` gives posterior
probabilities per component: a galaxy assigned 60% starburst / 40%
star-forming is far more informative than a hard starburst label, and
the uncertainty can propagate through downstream distance or luminosity
calculations. K-means gives no such uncertainty estimate.

In this case, both methods struggle with the star-forming/starburst
boundary for the same reason: the physical overlap between the two classes
is real and irreducible from these features. The 97% accuracy from the
semi-supervised KNN (using only 150 seeds) confirms that with enough
labelled examples the boundary can be drawn well, but unsupervised methods
face an intrinsic ceiling set by the genuine cluster overlap.

---
## 9. References

1. MacQueen, J. (1967). Some methods for classification and analysis
   of multivariate observations. *Proceedings of the 5th Berkeley
   Symposium on Mathematical Statistics and Probability*, 1, 281-297.
   Original K-means algorithm paper.

2. Dempster, A.P., Laird, N.M., and Rubin, D.B. (1977). Maximum likelihood
   from incomplete data via the EM algorithm. *Journal of the Royal
   Statistical Society, Series B*, 39(1), 1-38.
   Foundation paper for the EM algorithm used to fit Gaussian mixture models.

3. Noeske, K.G. et al. (2007). Star formation in AEGIS field galaxies
   since z=1.1: the dominance of gradually declining star formation,
   and the main sequence of star-forming galaxies.
   *Astrophysical Journal Letters*, 660(1), L43.
   https://doi.org/10.1086/517926
   Defines the star-forming main sequence that motivates the SFR-mass
   feature relationships in this dataset.

4. Brinchmann, J. et al. (2004). The physical properties of star-forming
   galaxies in the low-redshift universe. *Monthly Notices of the Royal
   Astronomical Society*, 351(4), 1151-1179.
   https://doi.org/10.1111/j.1365-2966.2004.07881.x
   SDSS-based study defining the spectroscopic features (Dn4000, Hα EW)
   used in this dataset and establishing the bimodal galaxy population.

5. Blanton, M.R. and Moustakas, J. (2009). Physical properties and
   environments of nearby galaxies.
   *Annual Review of Astronomy and Astrophysics*, 47, 159-210.
   https://doi.org/10.1146/annurev-astro-082708-101734
   Comprehensive review of galaxy classification methods and the physical
   meaning of photometric and spectroscopic features used here.

6. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.
   *Journal of Machine Learning Research*, 12, 2825-2830.
   https://jmlr.org/papers/v12/pedregosa11a.html